# Hợp nhất và xuất dữ liệu after

**Võ Minh Huy — 2311110117**

Giữ platelets đã làm sạch và thêm hai kết quả làm mịn độc lập. Giữ đủ 299 hồ sơ và một nhãn chưa xác minh. Mã Boolean lưu bổ sung; các biến riêng vẫn giữ để diễn giải.

In [1]:
from pathlib import Path
import importlib.util
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
spec = importlib.util.spec_from_file_location("xuat_after", thu_muc / "python/22_xuat_after.py")
xuat_after = importlib.util.module_from_spec(spec)
spec.loader.exec_module(xuat_after)
file_after = xuat_after.export_python()
after = pd.read_csv(file_after)
pd.DataFrame({"SoHoSo":[len(after)], "SoCotGomMaHoSo":[after.shape[1]],
              "SoNhanChoXacMinh":[after["DEATH_EVENT"].isna().sum()],
              "SoThayDoiMean":[after["platelets"].ne(after["platelets_mean_5_nhom"]).sum()],
              "SoThayDoi10Pct":[after["platelets"].ne(after["platelets_dieu_chinh_10pct"]).sum()]})

,SoHoSo,SoCotGomMaHoSo,SoNhanChoXacMinh,SoThayDoiMean,SoThayDoi10Pct
0,299,18,1,299,10


## Đối chiếu hai môi trường

SQL phải được xuất trước. So sánh toàn bộ tổ hợp 17 cột bằng giá trị số chính xác, gồm ô trống và số lần xuất hiện. Loại khóa nội bộ MaHoSo khỏi phép so sánh vì hai môi trường đánh số khác nhau.

In [2]:
kiem_tra = xuat_after.compare_exports()
pd.DataFrame({"KetQua":[kiem_tra["status"]],
              "SoHoSoSQL":[kiem_tra["rows_sql"]],
              "SoHoSoPython":[kiem_tra["rows_python"]],
              "SoCotDuLieuDoiChieu":[kiem_tra["compared_columns"]],
              "SoGiaTriDoiChieu":[kiem_tra["compared_values"]],
              "SoDongSaiKhac":[kiem_tra["mismatching_rows_both_directions"]]})

,KetQua,SoHoSoSQL,SoHoSoPython,SoCotDuLieuDoiChieu,SoGiaTriDoiChieu,SoDongSaiKhac
0,PASS,299,299,17,5083,0


In [3]:
after[["MaHoSo", "platelets", "platelets_mean_5_nhom",
       "platelets_dieu_chinh_10pct", "MaNhomTuoi", "MaBoolean", "DEATH_EVENT"]].head()

,MaHoSo,platelets,platelets_mean_5_nhom,platelets_dieu_chinh_10pct,MaNhomTuoi,MaBoolean,DEATH_EVENT
0,1,265000.00,261884.04,265000.00,4,12,1.0
1,2,263358.03,261884.04,263358.03,2,8,1.0
2,3,162000.00,142198.04,162000.00,3,24,1.0
3,4,210000.00,261884.04,210000.00,2,9,1.0
4,5,327000.00,261884.04,327000.00,3,3,1.0


**Nhận xét:** Kết quả kiểm tra phải là PASS, 299 hồ sơ mỗi file, 18 cột gồm khóa, 5.083 giá trị đối chiếu và 0 dòng sai khác. Hai nhánh làm mịn không nối tiếp nhau. Nhãn trống không được tự suy đoán; dùng 298 hồ sơ khi phân tích cần nhãn. Chưa thực hiện thống kê mục 3f trong notebook này.